# 14. Logistic regression and classification probabilities

**Level:** Intermediate

**Prerequisites:** 11-13: splits and models.

[Course index](00_START_HERE.ipynb) | [Previous](13_Linear_Regression.ipynb) | [Next](15_Decision_Trees.ipynb)

**Topics:** Logistic regression, Classification, Sigmoid, Decision thresholds, Probabilities.

**Matching script:** `examples\14_Logistic_Regression.py`

**How to study:** read the notes, predict an example's output, run it, trace each change in data/shape/state, then complete the practice task. Every worked code cell contains its own imports and data and can run independently of other notebooks.

**Execution contract:** Python 3.11+, course dependencies, CPU only, synthetic/local data. No hosted API requests, model-weight downloads, dataset downloads, listening web servers, credentials, or real device changes. The temporary-file lesson cleans up its own files. Expected errors are caught narrowly and asserted; unexpected errors must remain visible.

## Concept notes and mental model

Despite its name, logistic regression is commonly a classifier. For a binary target it maps a
linear score through sigmoid to estimate the positive-class probability. Training usually
minimizes log loss with regularization. Multiclass variants can use a softmax formulation.

A probability is not a class label. A threshold converts a probability into a decision and
controls a tradeoff between missed positives and false alarms. Choose thresholds using
validation data and operational costs, not by optimizing on the final test set.

Scaling can help optimization when feature units differ. A linear decision boundary cannot
represent every nonlinear problem without feature transformations. Estimated probabilities may
need calibration and their interpretation depends on whether deployment resembles training.

## Mathematics or data contract

sigmoid(z)=1/(1+exp(-z)); threshold 0.5 predicts positive when estimated p >= 0.5.

## Important limitation

Synthetic separable data does not establish reliability for real outages or rare events.

Distinguish what is implemented and asserted below from what would need additional data, training, independent evaluation, authorization, or operational controls.

## Worked example 1: Train and inspect probabilities on a held-out set

**Purpose:** See the difference between scores and decisions.

### Read the code in this order
1. Generate binary labels and split with stratification.
2. Scale inside a training pipeline.
3. Fit logistic regression and inspect class-1 probabilities.
4. Check normalization and nontrivial held-out accuracy.

### Syntax and behavior to notice
- `fit(...)` learns model or preprocessing state from the supplied training data.
- `predict_proba(...)` returns estimated class probabilities in the estimator's class order.
- `assert` checks the stated behavior; a failed check raises AssertionError.
- A fixed random seed makes the chosen sampling reproducible within the environment.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
first probabilities: [0.893, 0.008, 0.165]
held-out accuracy: 0.853
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [1]:
import numpy as np
from sklearn.datasets import make_classification
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score
from sklearn.model_selection import train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

X, y = make_classification(n_samples=300, n_features=6, n_informative=4,
                           class_sep=1.5, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, stratify=y, random_state=42)
model = make_pipeline(StandardScaler(), LogisticRegression(max_iter=300))
model.fit(X_train, y_train)
probabilities = model.predict_proba(X_test)
labels = (probabilities[:, 1] >= 0.5).astype(int)
score = accuracy_score(y_test, labels)
assert score > 0.75
np.testing.assert_allclose(probabilities.sum(axis=1), 1)
print("first probabilities:", np.round(probabilities[:3, 1], 3).tolist())
print("held-out accuracy:", round(score, 3))

first probabilities: [0.893, 0.008, 0.165]
held-out accuracy: 0.853


### Interpret example 1

The result should demonstrate this purpose: **See the difference between scores and decisions.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Worked example 2: Check sigmoid confidence and class threshold

**Purpose:** Inspect probabilities without interpreting them as guaranteed correctness.

### Read the code in this order
1. Compute stable sigmoid values for fixed scores.
2. Threshold into decisions.
3. Verify symmetry and the exact zero-score boundary.

### Syntax and behavior to notice
- `assert` checks the stated behavior; a failed check raises AssertionError.

### Expected output
This is the output captured from executing the worked example in the recorded environment. Package versions and floating-point rounding can change numeric details; the assertions express the required shapes, invariants, and thresholds.

```text
probabilities: [0.119, 0.5, 0.881] labels: [0, 1, 1]
```

**Before running:** identify the input variables, which values are learned versus chosen, and what each assertion proves. A successful shape test does not by itself prove predictive quality. Training-only loss checks are labeled as such.

In [2]:
import numpy as np

scores = np.array([-2., 0., 2.])
probabilities = 1 / (1 + np.exp(-scores))
labels = (probabilities >= 0.5).astype(int)
assert labels.tolist() == [0, 1, 1]
assert np.isclose(probabilities[0] + probabilities[2], 1)
assert probabilities[1] == 0.5
print("probabilities:", np.round(probabilities, 3).tolist(), "labels:", labels.tolist())

probabilities: [0.119, 0.5, 0.881] labels: [0, 1, 1]


### Interpret example 2

The result should demonstrate this purpose: **Inspect probabilities without interpreting them as guaranteed correctness.**

Follow the assertions back to the exact data or state being checked. Where a model was fitted, distinguish training fit from held-out evaluation. Where input is rejected, inspect whether rejection happens before any intended mutation or tool execution. Do not delete a failed assertion to make an incorrect result look valid.

## Practice: extend the concept yourself

Evaluate thresholds 0.3 and 0.7 on a validation set and report both precision and recall.

1. Write your expected result before editing code.
2. Add one normal-case test and one boundary/failure-case test.
3. Keep training/validation/test responsibilities separate where relevant.
4. Report an honest failure or unknown result rather than a success-shaped fallback.

The task is intentionally not filled in for you; the worked examples above are complete and runnable. Add your own code cell for the extension.

## Understanding checkpoint
- Explain `Logistic regression` in your own words and give one example of when it is useful.
- Trace one worked example from input through transformation to returned or printed output.
- State this lesson's mathematical/data contract: sigmoid(z)=1/(1+exp(-z)); threshold 0.5 predicts positive when estimated p >= 0.5.
- Name the boundary of the demonstration: Synthetic separable data does not establish reliability for real outages or rare events.
- Describe one test that could catch a plausible but wrong implementation.

## Primary references and further reading

- `https://scikit-learn.org/stable/user_guide.html`

Research snapshot: **2026-10-17**. Foundational references need not be recent. Mutable library docs describe their current version; compare with the tested dependency versions in the course manifest. References are reading material, not runtime downloads.